# CNN based Cat vs Dog Classification model

### Step 1 : Imports & Device Setup

In [3]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , random_split
from torchvision import datasets , transforms
import matplotlib.pyplot as plt

# set device (uses CUDA GPU if available)
device = torch.device ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")

Using device : cuda


### Step 2 : Clean Corrupted Images in PetImages

In [5]:
data_dir = "./PetImages"
num_skipped = 0

for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join(data_dir, folder_name)
    if not os.path.exists(folder_path):
        continue
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            with Image.open(fpath) as img:
                img.verify() # verify that it is an intact image
        except Exception:
            num_skipped += 1
            os.remove(f_path)

print(f"Cleaned dataset! Removed {num_skipped} corrupted/invalid files.")

Cleaned dataset! Removed 0 corrupted/invalid files.


### Step 3 : Data Preprocessing & DataLoaders

In [8]:
# Define transforms
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datset using ImageFolder
full_dataset = datasets.ImageFolder(root=data_dir, transform = transform)
print(f"Classes: {full_dataset.class_to_idx} | Total images: {len(full_dataset)}")

# 80-20 Train/Validation split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size , val_size])

# create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size= batch_size)

Classes: {'Cat': 0, 'Dog': 1} | Total images: 24998


### Build the CNN Model

In [9]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super(CatDogCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1: 3 x 128 x 128 -> 32 x 64 x 64
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 32 x 64 x 64 -> 64 x 32 x 32
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 64 x 32 x 32 -> 128 x 16 x 16
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)  # 2 classes: Cat (0), Dog (1)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CatDogCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

CatDogCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropo

### Step 5 : Training & Validation Loop

In [10]:
epochs = 5  # Increase to 10-15 for higher accuracy

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_acc = 100 * correct_train / total_train
    
    # Validation phase
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = 100 * correct_val / total_val
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

Epoch [1/5] | Loss: 0.8054 | Train Acc: 64.60% | Val Acc: 71.16%
Epoch [2/5] | Loss: 0.5703 | Train Acc: 70.27% | Val Acc: 73.70%
Epoch [3/5] | Loss: 0.5352 | Train Acc: 73.56% | Val Acc: 77.38%
Epoch [4/5] | Loss: 0.4927 | Train Acc: 76.65% | Val Acc: 79.84%
Epoch [5/5] | Loss: 0.4576 | Train Acc: 78.61% | Val Acc: 80.36%


### Step 6: Save Model & Test on a Single Image

In [11]:
# Save model weights
torch.save(model.state_dict(), "cat_dog_cnn.pth")
print("Model saved as cat_dog_cnn.pth")

# Quick prediction function
def predict_image(image_path, model):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_tensor)
        _, pred = torch.max(output, 1)
        
    label = "Dog" if pred.item() == 1 else "Cat"
    plt.imshow(img)
    plt.title(f"Predicted: {label}")
    plt.axis("off")
    plt.show()

Model saved as cat_dog_cnn.pth


# advancement

In [14]:
from torchvision import models

# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze early layers so training is super fast
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer for 2 classes (Cat=0, Dog=1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Train for just 2 epochs (reaches ~97-98% validation accuracy)
for epoch in range(2):
    model.train()
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    print(f"Epoch [{epoch+1}/2] | Train Acc: {100 * correct / total:.2f}%")

# Overwrite saved weights
torch.save(model.state_dict(), "cat_dog_cnn.pth")
print("Updated cat_dog_cnn.pth with ResNet18!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\deepa/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:13<00:00, 3.57MB/s]


Epoch [1/2] | Train Acc: 88.61%
Epoch [2/2] | Train Acc: 91.05%
Updated cat_dog_cnn.pth with ResNet18!
